In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !pip install "transformers[torch]"
# !pip install peft
# !pip install evaluate

In [ ]:
import yaml
from pathlib import Path
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from sklearn.utils.class_weight import compute_class_weight
import evaluate
from datasets import load_from_disk

In [ ]:
# ------------------
# LOAD CONFIGURATION
# ------------------
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

ROOT = Path().resolve().parents[0]
HDFS_FILE_PATH = ROOT / config["HDFS_FILE_PATH"]
ANOMALY_LABEL_FILE_PATH = ROOT / config["ANOMALY_LABEL_FILE_PATH"]
HDFS_TEMPLATES_FILE_PATH = ROOT / config["HDFS_TEMPLATES_FILE_PATH"]
TOKENIZED_DATASET_PATH = ROOT / config["TOKENIZED_DATASET_PATH"]
TYPE_OF_DATA = config["TYPE_OF_DATA"]
NUM_TRAIN_EPOCHS = config["NUM_TRAIN_EPOCHS"]
TRAIN_BATCH_SIZE = config["TRAIN_BATCH_SIZE"]
EVAL_BATCH_SIZE = config["EVAL_BATCH_SIZE"]
LEARNING_RATE = config["LEARNING_RATE"]
TRAINER_CHECKPOINTS_PATH = ROOT / config["TRAINER_CHECKPOINTS_PATH"]
LORA_RANK = config["LORA_RANK"]
LORA_ALPHA = config["LORA_ALPHA"]
LORA_DROPOUT = config["LORA_DROPOUT"]
CLASS_WEIGHTS_ENABLED = config["CLASS_WEIGHTS_ENABLED"]
FOCAL_ENABLED = config["FOCAL_ENABLED"]
GAMMA = config["GAMMA"]
MODEL_NAME = config["MODEL_NAME"]

In [ ]:
# ----------------
# TRAIN-TEST SPLIT
# ----------------
ds = load_from_disk(TOKENIZED_DATASET_PATH)
train_ds, eval_ds = ds["train"], ds["val"]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Normal", 1: "Anomaly"},
    label2id={"Normal": 0, "Anomaly": 1},
)

In [ ]:
# --------------
# TRAINING SETUP
# --------------
args = TrainingArguments(
    output_dir=TRAINER_CHECKPOINTS_PATH,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=True,
)

lora_cfg = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["query", "value"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_CLS",
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, class_weights=None, focal=False, gamma=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal = focal
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs[0]

        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        ce_loss = torch.nn.functional.cross_entropy(logits, labels, weight=weight, reduction="none")

        if self.focal:
            pt = torch.exp(-ce_loss)
            loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        else:
            loss = ce_loss.mean()

        return (loss, outputs) if return_outputs else loss

In [ ]:
if CLASS_WEIGHTS_ENABLED:
    labels = np.array([int(x) for x in train_ds["label"]])
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(labels),
        y=labels
    )
    print(weights)
    class_weights = torch.tensor(weights, dtype=torch.float)
else:
    class_weights = None

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = CustomTrainer(
    model=model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu")),
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    focal=FOCAL_ENABLED,
    gamma=GAMMA
)

In [ ]:
trainer.train()

In [ ]:
# adapter_dir = os.path.join("output", "lora_adapter")
# trainer.model.save_pretrained(adapter_dir)
# tokenizer.save_pretrained(adapter_dir)
# print(f"\nLoRA adapter saved to: {adapter_dir}")

In [ ]:
# def predict_block(log_templates):
#     text = " [SEP] ".join(log_templates)
#     enc = tokenizer(text, truncation=True, max_length=256, return_tensors="pt").to(trainer.model.device)
#     with torch.no_grad():
#         out = trainer.model(**enc)
#         probs = torch.softmax(out.logits, dim=-1).squeeze(0).tolist()
#     pred_id = int(torch.tensor(probs).argmax().item())
#     return {"pred_label": model.config.id2label[pred_id], "probs": {model.config.id2label[i]: float(p) for i, p in enumerate(probs)}}
#
#
#
# example_seq = anomaly_row = eval_df[eval_df["LabelId"] == 1].iloc[0]["text"]
# print("Example prediction:", predict_block(example_seq))

In [ ]:
# eval_df[eval_df["LabelId"] == 1].head()

In [ ]:
# anomaly_row = eval_df[eval_df["LabelId"] == 1].iloc[0]["text"]
# print(anomaly_row)
#
# example_seq = eval_df.iloc[3]["text"]
# print(example_seq)